In [1]:
"""SESSION 25 — SAM2 (ViT-L, frozen image encoder) fine-tuned with 5-fold OOF CV.
 --6 epoch per fold for 12 hour session budget.
"""

from __future__ import annotations

# %% 1. Standard library ──────────────────────────────────────────────────────
import argparse
import csv
import gc
import json
import multiprocessing
import os
import random
import re
import shutil
import subprocess
import sys
import time
import venv
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SESSION_NAME     = "session25_sam2_frozen_encoder_cv_6ep"
RUNTIME_BUILD_ID = "SESSION25-SAM2-6EP-V9-20261018"
IMAGE_SUFFIXES   = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Image_Annotation_rebuilt (Roboflow export): augmented copies share the
# source stem before ".rf."; these offline transform prefixes mark variants.
IMAGE_ANNOTATION_CLASSES = ["Nail_psoriasis", "SJS-TEN", "acne", "vitiligo"]
AUGMENT_PREFIX = re.compile(
    r"^(?:"
    r"Crop-(?:100450_100400|100450_0300|100350_100300|0350_100400|0350_0300)|"
    r"padd-(?:100100_100100|1000_00|0100_00|00_1000|00_0100)|"
    r"Resize-450_400|"
    r"flip-(?:-1|0|1)|"
    r"(?:hue|saturation)-(?:200|150|100|50)|"
    r"superpixels-(?:200|100|75|50|25)"
    r")"
)

# SAM2 checkpoint
SAM2_CHECKPOINT_URL = (
    "https://dl.fbaipublicfiles.com/segment_anything_2/092824/"
    "sam2.1_hiera_large.pt"
)
SAM2_HF_REPO   = "facebook/sam2.1-hiera-large"

ISOLATED_ENV    = Path("/kaggle/working/session25_sam2_env")
ISOLATED_PYTHON = ISOLATED_ENV / "bin" / "python"

# Training hyper-parameters (overrideable by CLI)
DEFAULT_EPOCHS       = 6       # per fold; ~34 min/epoch on a T4 with ~6,400 train images,
                               # so 5 folds sharded over 2 GPUs take ~11 h
DEFAULT_LR_DECODER   = 1e-5
DEFAULT_LR_PROMPT    = 1e-6
DEFAULT_WEIGHT_DECAY = 4e-5
DEFAULT_ACCUM_STEPS  = 8       # effective batch = 8 images
DEFAULT_SEED         = 42
MAX_SESSION_SECONDS  = 41_400  # 11.5 h -- Kaggle GPU cap is 12 h; keep 30 min margin


# ─────────────────────────────────────────────────────────────────────────────
# %% 2. Dependency bootstrap (isolated venv)
# ─────────────────────────────────────────────────────────────────────────────
def _pip_run(python: Path, *args: str) -> None:
    cmd = [str(python), "-m", "pip", "install", "-q", *args]
    print("+", " ".join(str(a) for a in args[:6]), "...", flush=True)
    subprocess.check_call(cmd)


# Probe: one-liner that only checks imports -- no try/except (invalid after semicolon)
# sam2 from GitHub has no __version__; we just verify the import + build_sam2 resolve
DEPENDENCY_PROBE_CODE = (
    "import torch, torchvision, sam2; "
    "from sam2.build_sam import build_sam2; "
    "print('torch=', torch.__version__, "
    "'cuda=', torch.cuda.is_available(), "
    "'gpus=', torch.cuda.device_count(), "
    "'sam2=OK')"
)


def _probe(python: Path | str) -> tuple[bool, str]:
    """Returns (ok, detail). Never raises -- returns (False, msg) on any error."""
    py_str = str(python)
    # Fast-fail: if the binary file doesn't exist, skip subprocess entirely
    if not Path(py_str).exists():
        return False, f"binary not found: {py_str}"
    try:
        # Cold `import torch` on Kaggle's filesystem can exceed 60 s; allow 5 min.
        result = subprocess.run(
            [py_str, "-u", "-c", DEPENDENCY_PROBE_CODE],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            timeout=300,
        )
        return result.returncode == 0, result.stdout.strip()
    except (FileNotFoundError, OSError, subprocess.TimeoutExpired) as exc:
        return False, f"{type(exc).__name__}: {exc}"


def _pip_run_verbose(python: Path, *args: str) -> None:
    """Like _pip_run but prints full output if pip fails."""
    cmd = [str(python), "-m", "pip", "install", *args]
    print("+", " ".join(str(a) for a in list(args)[:6]), "...", flush=True)
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        print(result.stdout[-3000:], flush=True)  # last 3000 chars of pip output
        raise RuntimeError(f"pip install failed (exit {result.returncode})")
    # Show only the last few lines so progress is visible
    lines = [l for l in result.stdout.splitlines() if l.strip()]
    for l in lines[-4:]:
        print("  ", l, flush=True)


def _install_isolated_env() -> None:
    work = Path("/kaggle/working")
    if not work.is_dir():
        raise RuntimeError("Not running on Kaggle -- /kaggle/working missing.")

    free_gib = shutil.disk_usage(work).free / 1024 ** 3
    if free_gib < 8:
        raise RuntimeError(
            f"Need >=8 GiB free in /kaggle/working; found {free_gib:.1f} GiB."
        )

    print(f"  [env] creating isolated SAM2 environment at {ISOLATED_ENV}", flush=True)
    # Reuse Kaggle's CUDA-enabled torch/torchvision instead of downloading a second
    # multi-gigabyte CUDA runtime into the isolated environment.
    try:
        venv.EnvBuilder(with_pip=True, clear=True, system_site_packages=True).create(ISOLATED_ENV)
    except Exception as err:
        print(f"  [env] stdlib venv failed ({err}); trying virtualenv ...", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        subprocess.check_call([
            sys.executable, "-m", "virtualenv",
            "--clear", "--no-download", "--system-site-packages",
            str(ISOLATED_ENV),
        ])

    py = ISOLATED_PYTHON

    # Install wrapt first -- silences Kaggle's sitecustomize ModuleNotFoundError
    _pip_run_verbose(py, "-q", "--upgrade", "pip", "setuptools", "wheel", "wrapt")

    # Core utilities
    _pip_run_verbose(
        py, "-q", "--upgrade",
        # Do NOT pin numpy: the venv reuses Kaggle's torch via system-site-packages,
        # and a numpy<2 pin breaks binaries built against numpy 2.x.
        "numpy",
        "opencv-python-headless",
        "Pillow", "pycocotools",
        "matplotlib", "tqdm",
        "huggingface_hub",
        "hydra-core", "PyYAML",
    )

    # iopath (legacy setup.py build) -- needs --no-build-isolation on new pip
    _pip_run_verbose(py, "-q", "--no-build-isolation", "iopath==0.1.10")

    # SAM2 from GitHub -- 'sam-2' does NOT exist on PyPI
    # facebook/sam2 is the correct repo; SAM2_BUILD_CUDA=0 skips CUDA ext build
    print("  [env] installing SAM2 from GitHub (CUDA extensions skipped) ...", flush=True)
    sam2_env = {**os.environ, "SAM2_BUILD_CUDA": "0", "PYTHONNOUSERSITE": "1"}
    sam2_cmd = [
        str(py), "-m", "pip", "install", "-q",
        "--no-build-isolation",
        "git+https://github.com/facebookresearch/sam2.git",
    ]
    result = subprocess.run(
        sam2_cmd, env=sam2_env,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        print("  [env] SAM2 git install failed; trying segment-anything-2 from PyPI ...", flush=True)
        print(result.stdout[-2000:], flush=True)
        # Fallback: PyPI package is named 'segment-anything-2' on some indexes
        alt_result = subprocess.run(
            [str(py), "-m", "pip", "install", "-q", "--no-build-isolation",
             "segment-anything-2"],
            env=sam2_env,
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        )
        if alt_result.returncode != 0:
            print(alt_result.stdout[-2000:], flush=True)
            raise RuntimeError(
                "Could not install SAM2 from GitHub or PyPI. "
                "Check internet connectivity (Kaggle: enable Internet in Settings).\n"
                + alt_result.stdout[-500:]
            )
    else:
        print(f"  [env] SAM2 installed OK", flush=True)

    ok, detail = _probe(py)
    if ok:
        print(f"  [env] full probe OK: {detail}", flush=True)
    else:
        raise RuntimeError(f"Isolated env probe failed after install:\n{detail}")


def _runtime_source() -> str:
    """Recover this script's source text for isolated re-launch."""
    candidate = globals().get("__file__")
    if candidate:
        try:
            text = Path(candidate).read_text(encoding="utf-8")
            if RUNTIME_BUILD_ID in text and "def main(" in text:
                return text
        except OSError:
            pass
    # Prefer the cell the user just ran over any stale /kaggle/working copy.
    try:
        history = get_ipython().user_ns.get("In", [])  # type: ignore
        for text in reversed(history):
            if isinstance(text, str) and RUNTIME_BUILD_ID in text and "def main(" in text:
                return text
    except Exception:
        pass
    cached = Path("/kaggle/working/SESSION_25_runtime.py")
    try:
        text = cached.read_text(encoding="utf-8")
        if RUNTIME_BUILD_ID in text and "def main(" in text:
            return text
    except OSError:
        pass
    raise RuntimeError(
        "Cannot recover source for re-launch. "
        "Make sure the script file exists in /kaggle/working/ "
        f"and contains the build-id {RUNTIME_BUILD_ID!r}."
    )


def _fold_dir_name(parts) -> str | None:
    for part in parts:
        if re.fullmatch(r"fold_\d+", part):
            return part
    return None


def _resume_target(parts, preds_dir: Path, work: Path) -> Path | None:
    name = parts[-1]
    if name.startswith("fold_") and name.endswith(".json"):
        return preds_dir / name
    fold_dir = _fold_dir_name(parts[:-1])
    if fold_dir and name in ("best_epoch.pth", "train_complete.json"):
        return work / fold_dir / name
    return None


def _import_resume_artifacts(src: Path, preds_dir: Path, work: Path) -> int:
    """Restore fold JSONs and trained checkpoints from a results zip or directory."""
    imported = 0
    if src.is_file() and src.suffix == ".zip":
        with zipfile.ZipFile(src) as zf:
            for member in zf.namelist():
                target = _resume_target(Path(member).parts, preds_dir, work)
                if target is not None:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(zf.read(member))
                    imported += 1
    elif src.is_dir():
        for path in src.rglob("*"):
            if path.is_file():
                target = _resume_target(path.relative_to(src).parts, preds_dir, work)
                if target is not None:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(path, target)
                    imported += 1
    return imported


def _forwarded_args(args: argparse.Namespace) -> list[str]:
    """CLI flags for a child process (isolated relaunch or GPU shard)."""
    cli = [
        "--data-root", args.data_root,
        "--folds", args.folds,
        "--epochs", str(args.epochs),
        "--seed", str(args.seed),
        "--lr-decoder", str(args.lr_decoder),
        "--lr-prompt", str(args.lr_prompt),
        "--weight-decay", str(args.weight_decay),
        "--accum-steps", str(args.accum_steps),
        "--image-size", str(args.image_size),
        "--device", args.device,
        "--gpu-shards", str(args.gpu_shards),
    ]
    if args.work_dir:
        cli += ["--work-dir", args.work_dir]
    for resume in args.resume_from:
        cli += ["--resume-from", resume]
    if args.accept_checkpoint_folds:
        cli += ["--accept-checkpoint-folds", args.accept_checkpoint_folds]
    if args.quick:
        cli.append("--quick")
    return cli


def _run_gpu_shards(args: argparse.Namespace, folds: list[int], work: Path,
                    preds_dir: Path, n_shards: int) -> None:
    """Train folds in parallel, one child process per GPU, then merge fold files."""
    import threading
    runner = Path("/kaggle/working/SESSION_25_runtime.py")
    runner.write_text(_runtime_source(), encoding="utf-8")

    children = []
    for gpu in range(n_shards):
        shard_folds = folds[gpu::n_shards]
        if not shard_folds:
            continue
        child_args = argparse.Namespace(**{
            **vars(args),
            "folds": ",".join(map(str, shard_folds)),
            "work_dir": str(work / f"shard_{gpu}"),
            "gpu_shards": 1,
        })
        cmd = [str(sys.executable), "-u", str(runner), *_forwarded_args(child_args)]
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
        print(f"  [shard] GPU {gpu} -> folds {shard_folds}", flush=True)
        proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True)
        children.append((gpu, proc))

    def pump(gpu: int, proc: subprocess.Popen) -> None:
        for line in proc.stdout:
            sys.stdout.write(f"[gpu{gpu}] {line}")
            sys.stdout.flush()

    pumps = [threading.Thread(target=pump, args=(gpu, proc), daemon=True)
             for gpu, proc in children]
    for thread in pumps:
        thread.start()
    for gpu, proc in children:
        if proc.wait() != 0:
            print(f"  [shard] GPU {gpu} exited with an error", flush=True)
    for thread in pumps:
        thread.join()

    for gpu in range(n_shards):
        for artifact in (work / f"shard_{gpu}" / "predictions").glob("fold_*_*.json"):
            shutil.copy2(artifact, preds_dir / artifact.name)


def _ensure_dependencies(args: argparse.Namespace) -> int | None:
    """Bootstrap isolated env and re-launch if current Python lacks SAM2."""
    ok, detail = _probe(sys.executable)
    if ok:
        print(f"  [deps] environment OK: {detail}")
        return None

    print("  [deps] current Python missing SAM2 -- bootstrapping ...")
    print(f"  [deps] probe output:\n  {detail}")

    # Check isolated env only if it actually exists (first run: go straight to install)
    iso_ok = False
    iso_detail = "not yet created"
    if ISOLATED_PYTHON.exists():
        iso_ok, iso_detail = _probe(ISOLATED_PYTHON)
        if iso_ok:
            print(f"  [deps] reusing existing isolated env: {iso_detail}")

    if not iso_ok:
        print(f"  [deps] installing isolated SAM2 env (this takes ~10 min) ...", flush=True)
        _install_isolated_env()
        iso_ok, iso_detail = _probe(ISOLATED_PYTHON)
    if not iso_ok:
        raise RuntimeError("Isolated env probe failed after install:\n" + iso_detail)

    runtime_path = Path("/kaggle/working/SESSION_25_runtime.py")
    source = _runtime_source()
    runtime_path.write_text(source, encoding="utf-8")

    cmd = [str(ISOLATED_PYTHON), "-u", str(runtime_path), *_forwarded_args(args)]
    print("  [deps] re-launching in isolated Python ...", flush=True)
    log_path = Path("/kaggle/working/SESSION_25_console.log")
    print(f"  [deps] streaming console to {log_path}", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with open(log_path, "a", encoding="utf-8") as log_file:
        proc = subprocess.Popen(
            cmd, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            log_file.write(line)
        return proc.wait()


# ─────────────────────────────────────────────────────────────────────────────
# %% 3. Dataset utilities
# ─────────────────────────────────────────────────────────────────────────────
def _split_dirs(root: Path, split: str) -> tuple[Path, Path]:
    split_names = {
        "val": ("val", "valid", "validation"),
        "valid": ("valid", "val", "validation"),
        "validation": ("validation", "valid", "val"),
    }.get(split, (split,))
    for split_name in split_names:
        v2 = (root / "images" / split_name, root / "labels" / split_name)
        if all(p.is_dir() for p in v2):
            return v2
        legacy = (root / split_name / "images", root / split_name / "labels")
        if all(p.is_dir() for p in legacy):
            return legacy
    return root / split / "images", root / split / "labels"


def _load_class_names(root: Path) -> list[str]:
    """Read the ordered YOLO class names from the dataset's data.yaml."""
    config_path = root / "data.yaml"
    if not config_path.is_file():
        raise FileNotFoundError(
            f"Missing {config_path}; provide a YOLO data.yaml with the class names."
        )

    try:
        import yaml
    except ModuleNotFoundError as error:
        raise RuntimeError(
            "PyYAML is required to read data.yaml; install PyYAML and retry."
        ) from error
    document = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}
    names_value = document.get("names")
    if isinstance(names_value, dict):
        try:
            names_value = [
                names_value[key]
                for key in sorted(names_value, key=lambda value: int(value))
            ]
        except (TypeError, ValueError) as error:
            raise ValueError(
                f"{config_path} names mapping keys must be integer class IDs."
            ) from error
    if (not isinstance(names_value, list)
            or not names_value
            or not all(isinstance(name, str) and name.strip() for name in names_value)):
        raise ValueError(
            f"{config_path} must define names as an ordered, non-empty list or mapping."
        )
    declared_count = document.get("nc")
    if declared_count is not None and int(declared_count) != len(names_value):
        raise ValueError(
            f"{config_path} declares nc={declared_count}, but has "
            f"{len(names_value)} class names."
        )
    return [name.strip() for name in names_value]


def _is_dataset_root(root: Path) -> bool:
    if not (root / "data.yaml").is_file():
        return False
    return all(
        (image_dir.is_dir() and label_dir.is_dir())
        for image_dir, label_dir in (_split_dirs(root, split) for split in ("train", "val"))
    )


def _is_image_annotation_dataset(root: Path) -> bool:
    """Match Image_Annotation_rebuilt by its 4-class names in data.yaml."""
    yaml_path = root / "data.yaml"
    if not yaml_path.is_file():
        return False
    try:
        text = yaml_path.read_text(encoding="utf-8")
    except OSError:
        return False
    return all(
        re.search(rf"^\s*\d+\s*:\s*{re.escape(name)}\s*$", text, re.MULTILINE)
        or re.search(rf"^\s*-\s*{re.escape(name)}\s*$", text, re.MULTILINE)
        for name in IMAGE_ANNOTATION_CLASSES
    )


def _find_dataset_root(hint: str) -> Path:
    if hint != "auto":
        explicit = Path(hint).expanduser().resolve()
        if _is_dataset_root(explicit):
            print(f"  [data] explicit dataset: {explicit}", flush=True)
            return explicit
        if explicit.is_dir():
            matches = sorted({
                yaml_path.parent.resolve()
                for yaml_path in explicit.rglob("data.yaml")
                if _is_dataset_root(yaml_path.parent)
            })
            if len(matches) == 1:
                print(f"  [data] found dataset under explicit directory: {matches[0]}")
                return matches[0]
            if len(matches) > 1:
                choices = "\n".join(f"    - {match}" for match in matches)
                raise RuntimeError(
                    "Multiple complete datasets were found under --data-root. "
                    "Pass the dataset directory explicitly:\n" + choices
                )
        raise FileNotFoundError(
            f"--data-root is not a complete YOLO segmentation dataset: {explicit}"
        )

    kaggle_input = Path("/kaggle/input")
    search_roots = [kaggle_input] if kaggle_input.is_dir() else [Path.cwd()]
    candidates = set()
    for search_root in search_roots:
        if search_root.is_dir():
            for yaml_path in search_root.rglob("data.yaml"):
                root = yaml_path.parent.resolve()
                if _is_dataset_root(root):
                    candidates.add(root)
    if not candidates:
        raise FileNotFoundError(
            "No complete YOLO dataset was found. Attach the dataset in Kaggle "
            "(Add Data) and keep --data-root=auto, or pass its mounted path "
            "such as --data-root /kaggle/input/<dataset-slug>."
        )
    preferred = [path for path in candidates if _is_image_annotation_dataset(path)]
    if not preferred:
        preferred = [
            path for path in candidates
            if "".join(char for char in path.name.lower() if char.isalnum())
            in {"skindisnet2annotation", "imageannotationrebuilt"}
        ]
    if not preferred:
        preferred = []
        for path in candidates:
            manifest = path / "manifest.csv"
            if not manifest.is_file():
                continue
            with manifest.open(newline="", encoding="utf-8") as handle:
                columns = set(next(csv.reader(handle), []))
            if {"name", "split", "source", "augmented"} <= columns:
                preferred.append(path)
    pool = preferred if preferred else list(candidates)
    if len(pool) != 1:
        choices = "\n".join(f"    - {path}" for path in sorted(pool))
        raise RuntimeError(
            "Multiple complete datasets were found. Pass --data-root "
            "explicitly:\n" + choices
        )
    selected = pool[0]
    print(f"  [data] selected dataset: {selected}", flush=True)
    return selected


def _parse_label(lbl_path: Path, names: list[str]) -> list[dict]:
    anns = []
    try:
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            if cls_id < 0 or cls_id >= len(names):
                continue
            coords = list(map(float, parts[1:]))
            if len(coords) >= 4 and len(coords) % 2 == 0:
                xs = coords[0::2]
                ys = coords[1::2]
                x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
                anns.append({
                    "class_id": cls_id,
                    "class_name": names[cls_id],
                    "polygon": list(zip(xs, ys)),
                    "bbox_norm": [x1, y1, x2 - x1, y2 - y1],
                })
    except Exception:
        pass
    return anns


def _discover_dataset(
    root: Path, names: list[str],
) -> tuple[list[dict], list[dict], list[dict]]:
    splits_out: list[list[dict]] = []
    for split in ("train", "val", "test"):
        img_dir, lbl_dir = _split_dirs(root, split)
        items: list[dict] = []
        if img_dir.is_dir() and lbl_dir.is_dir():
            for img_path in sorted(img_dir.iterdir()):
                if img_path.suffix.lower() not in IMAGE_SUFFIXES:
                    continue
                lbl_path = lbl_dir / (img_path.stem + ".txt")
                if not lbl_path.exists():
                    continue
                anns = _parse_label(lbl_path, names)
                if not anns:
                    continue
                items.append({
                    "image_path": str(img_path),
                    "label_path": str(lbl_path),
                    "annotations": anns,
                    "split": "valid" if split == "val" else split,
                })
        splits_out.append(items)
    return splits_out[0], splits_out[1], splits_out[2]


def _apply_source_manifest(root: Path, items: list[dict]) -> dict | None:
    """Attach source IDs from SkinDisNet or the older Session-00 manifest."""
    source_manifest = root / "manifest.csv"
    v2_manifest = root / "metadata" / "manifest.csv"
    if source_manifest.is_file():
        with source_manifest.open(newline="", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            required = {"name", "split", "source", "augmented"}
            if not reader.fieldnames or not required <= set(reader.fieldnames):
                raise ValueError(
                    f"{source_manifest} must contain columns {sorted(required)}"
                )
            lookup = {}
            for row in reader:
                split = row.get("split", "").strip().lower()
                split = "valid" if split in {"val", "validation"} else split
                key = (split, Path(row.get("name", "")).stem)
                source = row.get("source", "").strip()
                if not key[1] or not source or key in lookup:
                    raise ValueError("SkinDisNet source manifest is incomplete or ambiguous")
                lookup[key] = source
        manifest_kind = "SkinDisNet source manifest"
        item_key = lambda item: (item.get("split", ""), Path(item["image_path"]).stem)
    elif v2_manifest.is_file():
        with v2_manifest.open(newline="", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            required = {"image_file", "split_group_id"}
            if not reader.fieldnames or not required <= set(reader.fieldnames):
                raise ValueError(f"{v2_manifest} must contain columns {sorted(required)}")
            lookup = {
                Path(row["image_file"]).name: row["split_group_id"]
                for row in reader if row.get("image_file")
            }
        manifest_kind = "V2 image group manifest"
        item_key = lambda item: Path(item["image_path"]).name
    else:
        return None

    missing = []
    for item in items:
        source = lookup.get(item_key(item))
        if not source:
            missing.append(item["image_path"])
        else:
            item["group_id"] = source
    if missing:
        raise ValueError(
            f"{manifest_kind} is missing {len(missing)} image entries; "
            f"first missing: {missing[0]}"
        )
    return {
        "manifest_type": manifest_kind,
        "source_group_count": len({item["group_id"] for item in items}),
    }


def _apply_filename_groups(items: list[dict]) -> dict | None:
    """Group Roboflow ".rf." variants by source stem when no manifest exists.

    Sets group_id on every item, marks excluded class-conflict groups, and
    flags one original-preferred image per group for validation. Returns a
    stats dict, or None when the dataset is not a Roboflow-style export.
    """
    if not items or not any(".rf." in Path(i["image_path"]).stem for i in items):
        return None
    groups: dict[str, list[dict]] = {}
    for item in items:
        raw_stem = Path(item["image_path"]).stem.split(".rf.", 1)[0]
        source = AUGMENT_PREFIX.sub("", raw_stem, count=1)
        item["group_id"] = source
        item["augmented_filename"] = bool(AUGMENT_PREFIX.match(raw_stem))
        item["validation_image"] = False
        groups.setdefault(source, []).append(item)

    def class_key(item: dict) -> tuple:
        return tuple(sorted({a["class_id"] for a in item["annotations"]}))

    conflicts = {
        group for group, members in groups.items()
        if len({class_key(m) for m in members}) > 1
    }
    for item in items:
        item["excluded"] = item["group_id"] in conflicts
    kept = [item for item in items if not item["excluded"]]
    by_group: dict[str, list[dict]] = {}
    for item in kept:
        by_group.setdefault(item["group_id"], []).append(item)
    for members in by_group.values():
        canonical = [m for m in members if not m["augmented_filename"]]
        min(canonical or members, key=lambda m: m["image_path"])[
            "validation_image"] = True
    return {
        "manifest_type": "filename-derived source groups (no manifest)",
        "source_group_count": len(by_group),
        "excluded_conflict_groups": len(conflicts),
        "cross_split_groups": sum(
            len({m["split"] for m in members}) > 1
            for members in by_group.values()
        ),
        "kept_images": len(kept),
    }


def _item_classes(item: dict) -> list[int]:
    return [a["class_id"] for a in item.get("annotations", [])]


def _make_folds(items: list[dict], num_classes: int, n_splits: int, seed: int):
    """Stratified folds, grouping manifest-linked variants when available."""
    import numpy as np

    if items and all("group_id" in item for item in items):
        groups: dict[str, list[int]] = {}
        for index, item in enumerate(items):
            groups.setdefault(item["group_id"], []).append(index)
        if len(groups) < n_splits:
            raise ValueError(
                f"{n_splits}-fold CV requires at least {n_splits} independent source groups"
            )
        group_ids = sorted(groups)
        label_matrix = np.zeros((len(items), num_classes), dtype=np.int32)
        for index, item in enumerate(items):
            for class_id in _item_classes(item):
                if 0 <= class_id < num_classes:
                    label_matrix[index, class_id] = 1
        features = np.asarray([
            [int(label_matrix[groups[group_id], class_id].sum())
             for class_id in range(num_classes)] + [len(groups[group_id])]
            for group_id in group_ids
        ], dtype=float)
        target = np.maximum(features.sum(axis=0) / n_splits, 1.0)
        best_assignment = None
        best_objective = float("inf")
        for trial in range(32):
            rng = random.Random(seed + trial)
            order = list(range(len(group_ids)))
            rng.shuffle(order)
            order.sort(key=lambda index: -float((features[index] / target).max()))
            counts = np.zeros((n_splits, num_classes + 1), dtype=float)
            assignment = {}
            for index in order:
                costs = []
                for fold_id in range(n_splits):
                    projected = counts.copy()
                    projected[fold_id] += features[index]
                    costs.append(float(((projected / target) ** 2).sum()))
                minimum = min(costs)
                choices = [
                    fold_id for fold_id, cost in enumerate(costs)
                    if abs(cost - minimum) < 1e-10
                ]
                selected_fold = rng.choice(choices)
                counts[selected_fold] += features[index]
                assignment[group_ids[index]] = selected_fold
            objective = int((counts[:, :num_classes] == 0).sum()) * 1e6 + float(
                (((counts - target) / target) ** 2).sum()
            )
            if objective < best_objective:
                best_objective = objective
                best_assignment = assignment
        assert best_assignment is not None
        folds = []
        required_classes = set(range(num_classes))
        for fold_id in range(n_splits):
            train = [
                item for item in items
                if best_assignment[item["group_id"]] != fold_id
            ]
            valid = [
                item for item in items
                if best_assignment[item["group_id"]] == fold_id
                and item.get("validation_image", True)
            ]
            if not train or not valid:
                raise ValueError(f"Fold {fold_id} is empty after source-group assignment")
            train_groups = {item["group_id"] for item in train}
            valid_groups = {item["group_id"] for item in valid}
            if train_groups & valid_groups:
                raise AssertionError("A source group crossed a CV fold boundary")
            held_out_groups = {
                item["group_id"] for item in items
                if best_assignment[item["group_id"]] == fold_id
            }
            if valid_groups != held_out_groups:
                raise AssertionError(
                    f"Fold {fold_id} has held-out groups without a validation image"
                )
            for split_name, split_items in (("train", train), ("valid", valid)):
                present = {
                    class_id for item in split_items for class_id in _item_classes(item)
                }
                missing = required_classes - present
                if missing and split_name == "train":
                    raise ValueError(
                        f"Fold {fold_id} train split lacks classes {missing}"
                    )
                if missing:
                    print(
                        f"  [warn] fold {fold_id} validation lacks classes "
                        f"{sorted(missing)}",
                        flush=True,
                    )
            folds.append({
                "fold": fold_id,
                "train": train,
                "valid": valid,
            })
        return folds

    dominant = np.array([max(set(_item_classes(it)), key=_item_classes(it).count)
                         if _item_classes(it) else 0 for it in items])
    rng = np.random.RandomState(seed)
    n = len(items)
    fold_of = np.full(n, -1, dtype=np.int32)
    for cls in np.unique(dominant):
        idx = np.where(dominant == cls)[0]
        rng.shuffle(idx)
        for k, chunk in enumerate(np.array_split(idx, n_splits)):
            for i in chunk:
                if fold_of[i] < 0:
                    fold_of[i] = k % n_splits
    for k, i in enumerate(np.where(fold_of < 0)[0]):
        fold_of[i] = k % n_splits
    folds = []
    for fold in range(n_splits):
        folds.append({
            "fold": fold,
            "train": [it for j, it in enumerate(items) if fold_of[j] != fold],
            "valid": [it for j, it in enumerate(items) if fold_of[j] == fold],
        })
    return folds


# ─────────────────────────────────────────────────────────────────────────────
# %% 4. SAM2 model build and freeze
# ─────────────────────────────────────────────────────────────────────────────
def _download_checkpoint(cache_dir: Path) -> Path:
    cache_dir.mkdir(parents=True, exist_ok=True)
    dest = cache_dir / "sam2.1_hiera_large.pt"
    if dest.exists() and dest.stat().st_size > 100_000_000:
        print(f"  [ckpt] reusing cached checkpoint ({dest.stat().st_size/1e9:.2f} GB)", flush=True)
        return dest
    print("  [ckpt] downloading SAM2.1-Hiera-Large ...", flush=True)
    try:
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(
            repo_id=SAM2_HF_REPO,
            filename="sam2.1_hiera_large.pt",
            local_dir=str(cache_dir),
        )
        if Path(src) != dest:
            shutil.copy2(src, dest)
    except Exception as hf_err:
        print(f"  [ckpt] HuggingFace failed ({hf_err}); falling back to FBai CDN ...", flush=True)
        subprocess.check_call(["wget", "-q", "-O", str(dest), SAM2_CHECKPOINT_URL])
    print(f"  [ckpt] OK ({dest.stat().st_size/1e9:.2f} GB)", flush=True)
    return dest


def _build_sam2_model(checkpoint: Path, device: str):
    """Build SAM2-Hiera-Large; return (model, predictor).

    build_sam2() uses Hydra and expects a *config name* relative to the sam2
    package's configs directory (e.g. 'sam2.1/sam2.1_hiera_l.yaml'), NOT an
    absolute filesystem path.  Passing a full path causes Hydra to strip the
    leading '/' and then fail with MissingConfigException.
    """
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    import sam2 as _sam2_pkg

    pkg_root = Path(_sam2_pkg.__file__).parent

    # Verify config exists on disk for a clear error message
    cfg_abs = pkg_root / "configs" / "sam2.1" / "sam2.1_hiera_l.yaml"
    if not cfg_abs.exists():
        cfg_abs = pkg_root.parent / "configs" / "sam2.1" / "sam2.1_hiera_l.yaml"
    if not cfg_abs.exists():
        raise FileNotFoundError(
            f"SAM2 config not found near {pkg_root}.\n"
            "Reinstall: pip install git+https://github.com/facebookresearch/sam2.git"
        )

    # IMPORTANT: build_sam2 expects a RELATIVE config name (resolved via Hydra
    # pkg://sam2), NOT an absolute path. Passing a full path strips the leading
    # '/' and produces: MissingConfigException: Cannot find 'kaggle/working/...'
    config_name = "sam2.1/sam2.1_hiera_l.yaml"
    print(f"  [model] building SAM2 (config={config_name}) ...", flush=True)

    try:
        model = build_sam2(config_name, str(checkpoint), device=device)
    except Exception as hydra_err:
        # Fallback: initialize Hydra with the explicit config directory
        print(f"  [model] default build failed ({hydra_err!r}); trying config_dir init ...", flush=True)
        from hydra._internal.utils import create_automatic_config_search_path
        from hydra.core.global_hydra import GlobalHydra
        from hydra.core.plugins import Plugins
        from hydra.core.utils import JobReturn
        GlobalHydra.instance().clear()
        # Re-try with the absolute config directory registered
        import sam2.build_sam as _bsam
        orig_compose = None
        try:
            from hydra import initialize_config_dir, compose
            GlobalHydra.instance().clear()
            with initialize_config_dir(
                config_dir=str(cfg_abs.parent.parent),  # points to sam2/configs/
                version_base="1.2",
            ):
                model = build_sam2(
                    "sam2.1/sam2.1_hiera_l.yaml",
                    str(checkpoint), device=device,
                )
        finally:
            GlobalHydra.instance().clear()

    predictor = SAM2ImagePredictor(model)
    return model, predictor


def _freeze_image_encoder(model) -> int:
    frozen = 0
    model.image_encoder.eval()
    for p in model.image_encoder.parameters():
        p.requires_grad_(False)
        frozen += p.numel()
    # Freeze memory modules (not used for single-image fine-tuning)
    for attr in ("memory_attention", "memory_encoder",
                 "obj_ptr_proj", "obj_ptr_tpos_proj", "obj_ptr_to_image_proj"):
        if hasattr(model, attr):
            mod = getattr(model, attr)
            mod.eval()
            for p in mod.parameters():
                p.requires_grad_(False)
                frozen += p.numel()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  [model] frozen={frozen/1e6:.1f}M  trainable={trainable/1e6:.1f}M params", flush=True)
    return frozen


# ─────────────────────────────────────────────────────────────────────────────
# %% 5. Data helpers
# ─────────────────────────────────────────────────────────────────────────────
def _polygon_to_mask(polygon_norm: list, img_w: int, img_h: int):
    import numpy as np
    import cv2
    pts = np.array(
        [(int(x * img_w), int(y * img_h)) for x, y in polygon_norm],
        dtype=np.int32,
    ).reshape((-1, 1, 2))
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 1)
    return mask


def _load_image_and_anns(item: dict):
    """Returns (image_np HWC uint8, boxes_xyxy float32 Nx4, masks list[HxW uint8], class_ids list[int])."""
    import numpy as np
    import cv2
    img_bgr = cv2.imread(item["image_path"])
    if img_bgr is None:
        return None, None, None, None
    img_np = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W = img_np.shape[:2]
    boxes, masks, cids = [], [], []
    for ann in item["annotations"]:
        mask_hw = _polygon_to_mask(ann["polygon"], W, H)
        ys, xs = np.where(mask_hw)
        if len(xs) == 0:
            continue
        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        boxes.append([float(x1), float(y1), float(x2), float(y2)])
        masks.append(mask_hw)
        cids.append(ann["class_id"])
    if not boxes:
        return img_np, None, None, None
    return img_np, np.array(boxes, dtype=np.float32), masks, cids


# ─────────────────────────────────────────────────────────────────────────────
# %% 6. Loss functions
# ─────────────────────────────────────────────────────────────────────────────
def _focal_loss(logits, targets, alpha=0.8, gamma=2.0):
    import torch, torch.nn.functional as F
    bce = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction="none")
    pt  = torch.exp(-bce)
    return (alpha * (1 - pt) ** gamma * bce).mean()


def _dice_loss(logits, targets, smooth=1.0):
    import torch
    pred  = torch.sigmoid(logits)
    inter = (pred * targets.float()).sum(dim=(-2, -1))
    denom = pred.sum(dim=(-2, -1)) + targets.float().sum(dim=(-2, -1)) + smooth
    return (1 - (2 * inter + smooth) / denom).mean()


def _seg_loss(logits, targets):
    return 0.20 * _focal_loss(logits, targets) + 0.80 * _dice_loss(logits, targets)


# ─────────────────────────────────────────────────────────────────────────────
# %% 7. SAM2 forward pass (batched box prompts for one image)
# ─────────────────────────────────────────────────────────────────────────────
def _sam2_forward(
    predictor, image_np, boxes_xyxy, device, image_already_set: bool = False,
):
    """
    Runs SAM2 on one image with N box prompts.
    Image encoder runs under torch.no_grad() (frozen).
    Prompt encoder + mask decoder produce gradients.
    Returns:
        logits_full  (N, 1, H_orig, W_orig)  -- full-resolution logits
        iou_preds    (N, 1)                  -- IoU quality predictions
    """
    import torch
    import torch.nn.functional as F
    import numpy as np

    # Encode image (frozen -- no grad stored)
    if not image_already_set:
        with torch.no_grad():
            predictor.set_image(image_np)

    # Access internal cached features
    img_embed   = predictor._features["image_embed"][-1]    # (1, C, H_enc, W_enc)
    hi_feats    = predictor._features.get("high_res_feats", [])
    H_orig, W_orig = predictor._orig_hw[0]

    N = len(boxes_xyxy)
    boxes_resized = predictor._transforms.transform_boxes(
        torch.as_tensor(boxes_xyxy, dtype=torch.float32, device=device),
        normalize=True,
        orig_hw=(H_orig, W_orig),
    )
    box_corners = torch.as_tensor(
        boxes_resized, dtype=torch.float32, device=device
    ).reshape(N, 2, 2)

    # Prompt encoder (trains gradients)
    sparse_emb, dense_emb = predictor.model.sam_prompt_encoder(
        points=None,
        boxes=box_corners,   # (N, 2, 2) SAM2 resized-image coordinates
        masks=None,
    )

    # Expand shared image embedding from (1, ...) to (N, ...)
    img_n   = img_embed.expand(N, -1, -1, -1)
    hi_n    = [f.expand(N, -1, -1, -1) for f in hi_feats]

    # Mask decoder (trains gradients)
    low_res_logits, iou_preds, _, _ = predictor.model.sam_mask_decoder(
        image_embeddings=img_n,
        image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_emb,
        dense_prompt_embeddings=dense_emb,
        multimask_output=False,
        repeat_image=False,
        high_res_features=hi_n if hi_n else None,
    )
    # low_res_logits: (N, 1, H_low, W_low) -- e.g. (N, 1, 256, 256)

    # Upsample to original image resolution for per-pixel loss
    logits_full = F.interpolate(
        low_res_logits, size=(H_orig, W_orig),
        mode="bilinear", align_corners=False,
    )   # (N, 1, H_orig, W_orig)

    return logits_full, iou_preds


# ─────────────────────────────────────────────────────────────────────────────
# %% 8. Training -- one fold
# ─────────────────────────────────────────────────────────────────────────────
def _train_fold(model, predictor, train_items: list[dict],
                fold_work: Path, args: argparse.Namespace,
                device: str, session_start: float) -> Path:
    import torch

    fold_work.mkdir(parents=True, exist_ok=True)
    rng = random.Random(args.seed)
    use_amp = device.startswith("cuda")

    optimizer = torch.optim.AdamW([
        {"params": model.sam_mask_decoder.parameters(),  "lr": args.lr_decoder},
        {"params": model.sam_prompt_encoder.parameters(), "lr": args.lr_prompt},
    ], weight_decay=args.weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=args.epochs, eta_min=args.lr_decoder * 0.01,
    )
    scaler    = torch.amp.GradScaler(device, enabled=use_amp)
    best_loss = float("inf")
    best_ckpt = fold_work / "best_epoch.pth"
    budget_hit = False

    for epoch in range(args.epochs):
        if time.time() - session_start > MAX_SESSION_SECONDS:
            print(f"  [time] {MAX_SESSION_SECONDS/3600:.1f}-h budget reached "
                  f"at epoch {epoch}; stopping.", flush=True)
            budget_hit = True
            break
        epoch_start = time.time()

        shuffled = train_items.copy()
        rng.shuffle(shuffled)
        epoch_loss = 0.0
        n_images   = 0
        optimizer.zero_grad()

        for item in shuffled:
            image_np, boxes_xyxy, gt_masks, class_ids = _load_image_and_anns(item)
            if image_np is None or boxes_xyxy is None:
                continue

            import torch as _t
            n_masks = len(gt_masks)
            image_loss = 0.0
            for mask_idx, gt_mask in enumerate(gt_masks):
                # Keep only one prompt's decoder activations on the GPU at a time.
                with _t.amp.autocast(device, enabled=use_amp):
                    logits, _iou = _sam2_forward(
                        predictor,
                        image_np,
                        boxes_xyxy[mask_idx:mask_idx + 1],
                        device,
                        image_already_set=mask_idx > 0,
                    )
                    gt_tensor = _t.as_tensor(
                        gt_mask[None, None, ...],
                        dtype=_t.float32,
                        device=device,
                    )
                    loss = (
                        _seg_loss(logits, gt_tensor)
                        / (args.accum_steps * n_masks)
                    )

                scaler.scale(loss).backward()
                image_loss += (
                    loss.detach().item() * args.accum_steps * n_masks
                )
                del logits, gt_tensor, loss

            epoch_loss  += image_loss / n_masks
            n_images    += 1

            if n_images % 200 == 0:
                print(
                    f"    epoch {epoch+1:3d}/{args.epochs}"
                    f"  {n_images}/{len(shuffled)} images"
                    f"  running_loss={epoch_loss/n_images:.4f}",
                    flush=True,
                )

            if n_images % args.accum_steps == 0:
                scaler.unscale_(optimizer)
                _t.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

        # flush remaining gradient accumulation
        if n_images % args.accum_steps != 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        scheduler.step()
        avg_loss = epoch_loss / max(n_images, 1)
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                "epoch": epoch,
                "mask_decoder_state":  model.sam_mask_decoder.state_dict(),
                "prompt_encoder_state": model.sam_prompt_encoder.state_dict(),
                "loss": best_loss,
            }, best_ckpt)

        epoch_seconds = time.time() - epoch_start
        print(
            f"    epoch {epoch+1:3d}/{args.epochs}"
            f"  loss={avg_loss:.4f}  best={best_loss:.4f}"
            f"  lr={scheduler.get_last_lr()[0]:.2e}"
            f"  {epoch_seconds/60:.1f} min/epoch",
            flush=True,
        )
        if epoch == 0:
            print(f"    [time] projected {epoch_seconds*args.epochs/3600:.1f} h "
                  f"for this fold at {args.epochs} epochs", flush=True)

    if not budget_hit:
        (fold_work / "train_complete.json").write_text(
            json.dumps({"epochs": args.epochs}))
    return best_ckpt


# ─────────────────────────────────────────────────────────────────────────────
# %% 9. OOF Validation -- generate COCO-format predictions
# ─────────────────────────────────────────────────────────────────────────────
def _validate_fold(model, predictor, checkpoint: Path,
                   val_items: list[dict], names: list[str],
                   device: str, img_id_offset: int = 0):
    import torch
    import numpy as np

    state = torch.load(checkpoint, map_location=device, weights_only=True)
    model.sam_mask_decoder.load_state_dict(state["mask_decoder_state"])
    model.sam_prompt_encoder.load_state_dict(state["prompt_encoder_state"])
    model.sam_mask_decoder.eval()
    model.sam_prompt_encoder.eval()

    coco_images, coco_anns = [], []
    ann_id = img_id_offset * 10_000

    with torch.no_grad():
        for rel_idx, item in enumerate(val_items):
            if rel_idx % 100 == 0:
                print(
                    f"    validating {rel_idx}/{len(val_items)} images",
                    flush=True,
                )
            img_id = img_id_offset + rel_idx
            image_np, boxes_xyxy, gt_masks, class_ids = _load_image_and_anns(item)
            if image_np is None or boxes_xyxy is None:
                continue

            H, W = image_np.shape[:2]
            coco_images.append({
                "id": img_id,
                "file_name": Path(item["image_path"]).name,
                "width": W, "height": H,
            })

            pred_masks = []
            scores = []
            for mask_idx in range(len(boxes_xyxy)):
                logits, iou_preds = _sam2_forward(
                    predictor,
                    image_np,
                    boxes_xyxy[mask_idx:mask_idx + 1],
                    device,
                    image_already_set=mask_idx > 0,
                )
                pred_masks.append(
                    (torch.sigmoid(logits) > 0.5)
                    .squeeze(1).squeeze(0).cpu().numpy()
                )
                scores.append(
                    torch.sigmoid(iou_preds).reshape(-1)[0].cpu().item()
                )
                del logits, iou_preds
            pred_bin = np.stack(pred_masks, axis=0)

            import pycocotools.mask as mask_util
            for n_i, (pred_mask, cls_id, score) in enumerate(
                zip(pred_bin, class_ids, scores)
            ):
                rle = mask_util.encode(np.asfortranarray(pred_mask.astype(np.uint8)))
                rle["counts"] = rle["counts"].decode("utf-8")
                b = boxes_xyxy[n_i].tolist()
                coco_anns.append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": cls_id + 1,
                    "segmentation": rle,
                    "bbox": [b[0], b[1], b[2] - b[0], b[3] - b[1]],
                    "area": float(mask_util.area(rle)),
                    "score": float(score),
                    "iscrowd": 0,
                })
                ann_id += 1

    return coco_images, coco_anns


# ─────────────────────────────────────────────────────────────────────────────
# %% 10. GT COCO builder + metric computation
# ─────────────────────────────────────────────────────────────────────────────
def _build_gt_coco(items: list[dict], names: list[str], img_id_offset: int = 0):
    import numpy as np, pycocotools.mask as mask_util
    gt_images, gt_anns = [], []
    ann_id = img_id_offset * 10_000
    for rel_idx, item in enumerate(items):
        img_id = img_id_offset + rel_idx
        image_np, boxes_xyxy, gt_masks, class_ids = _load_image_and_anns(item)
        if image_np is None or boxes_xyxy is None:
            continue
        H, W = image_np.shape[:2]
        gt_images.append({"id": img_id, "file_name": Path(item["image_path"]).name,
                           "width": W, "height": H})
        for mask_hw, cls_id, box in zip(gt_masks, class_ids, boxes_xyxy):
            rle = mask_util.encode(np.asfortranarray(mask_hw.astype(np.uint8)))
            rle["counts"] = rle["counts"].decode("utf-8")
            b = box.tolist()
            gt_anns.append({
                "id": ann_id, "image_id": img_id,
                "category_id": cls_id + 1,
                "segmentation": rle, "iscrowd": 0,
                "bbox": [b[0], b[1], b[2] - b[0], b[3] - b[1]],
                "area": float(mask_util.area(rle)),
            })
            ann_id += 1
    return gt_images, gt_anns


def _compute_coco_metrics(gt_images, gt_anns, pred_anns, names):
    gt_class_ids = {ann["category_id"] for ann in gt_anns}
    metrics = {"mAP50-95": 0.0, "mAP50": 0.0}
    metrics.update({
        f"mAP50_{name}": 0.0 if ci + 1 in gt_class_ids else -1.0
        for ci, name in enumerate(names)
    })
    if not pred_anns:
        return metrics

    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import io, contextlib

    gt_dict = {
        "info": {}, "licenses": [],
        "categories": [{"id": i + 1, "name": n} for i, n in enumerate(names)],
        "images": gt_images,
        "annotations": gt_anns,
    }
    coco_gt = COCO(); coco_gt.dataset = gt_dict; coco_gt.createIndex()
    coco_dt = coco_gt.loadRes(pred_anns)

    evaluator = COCOeval(coco_gt, coco_dt, "segm")
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
    print(buf.getvalue(), flush=True)

    stats   = evaluator.stats
    metrics["mAP50-95"] = float(stats[0])
    metrics["mAP50"] = float(stats[1])

    for ci, name in enumerate(names):
        ev_cat = COCOeval(coco_gt, coco_dt, "segm")
        ev_cat.params.catIds = [ci + 1]
        buf2 = io.StringIO()
        with contextlib.redirect_stdout(buf2):
            ev_cat.evaluate(); ev_cat.accumulate(); ev_cat.summarize()
        metrics[f"mAP50_{name}"] = float(ev_cat.stats[1])
    return metrics


def _print_map50_report(title: str, metrics: dict, names: list[str]) -> None:
    def display(value: float) -> str:
        return "N/A (no GT)" if value < 0 else f"{value:.4f}"

    print(f"  {title} segmentation mAP50: {display(metrics['mAP50'])}", flush=True)
    print("  Per-class segmentation mAP50:", flush=True)
    for name in names:
        print(f"    {name:40s} {display(metrics[f'mAP50_{name}'])}", flush=True)


# ─────────────────────────────────────────────────────────────────────────────
# %% 11. Main
# ─────────────────────────────────────────────────────────────────────────────
def main(argv=None) -> int:
    parser = argparse.ArgumentParser(
        description=f"SESSION 25: SAM2 frozen-encoder ({RUNTIME_BUILD_ID})"
    )
    parser.add_argument("--data-root",    default="auto")
    parser.add_argument("--work-dir",     default="")
    parser.add_argument("--folds",        default="0,1,2,3,4")
    parser.add_argument("--epochs",       type=int,   default=DEFAULT_EPOCHS)
    parser.add_argument("--seed",         type=int,   default=DEFAULT_SEED)
    parser.add_argument("--lr-decoder",   type=float, default=DEFAULT_LR_DECODER)
    parser.add_argument("--lr-prompt",    type=float, default=DEFAULT_LR_PROMPT)
    parser.add_argument("--weight-decay", type=float, default=DEFAULT_WEIGHT_DECAY)
    parser.add_argument("--accum-steps",  type=int,   default=DEFAULT_ACCUM_STEPS)
    parser.add_argument("--image-size",   type=int,   default=1024)
    parser.add_argument("--device",       default="auto")
    parser.add_argument("--resume-from",  action="append", default=[],
                        help="dir or results zip with prior fold_* artifacts "
                             "from an earlier Kaggle session (repeatable)")
    parser.add_argument("--accept-checkpoint-folds", default="",
                        help="comma-separated folds whose restored best_epoch.pth "
                             "is treated as fully trained (validate only)")
    parser.add_argument("--gpu-shards",   type=int, default=0,
                        help="0 = auto (one child process per GPU, folds split "
                             "across them); 1 = single process")
    parser.add_argument("--quick",        action="store_true",
                        help="2 epochs / fold -- smoke-test only")
    args, _ = parser.parse_known_args(argv)

    if args.quick:
        args.epochs = 2

    session_start = float(os.environ.setdefault("SESSION25_START", str(time.time())))

    print("=" * 76)
    print("SESSION 25 -- SAM2 ViT-L (frozen encoder) five-fold OOF CV")
    print(f"BUILD: {RUNTIME_BUILD_ID}")
    print(f"PYTHON: {sys.executable}")
    print("=" * 76, flush=True)
    print(
        "[args] "
        + json.dumps(
            {
                "data_root": args.data_root,
                "work_dir": args.work_dir or "(default)",
                "folds": args.folds,
                "epochs": args.epochs,
                "seed": args.seed,
                "image_size": args.image_size,
                "accum_steps": args.accum_steps,
                "device": args.device,
                "resume_from": args.resume_from or "(none)",
                "gpu_shards": args.gpu_shards,
                "quick": args.quick,
            }
        ),
        flush=True,
    )

    # ── Dependency bootstrap ──────────────────────────────────────────────
    relaunch_code = _ensure_dependencies(args)
    if relaunch_code is not None:
        return relaunch_code

    # ── (running inside isolated env from here) ───────────────────────────
    import torch, numpy as np, statistics as _stats

    if args.device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        device = args.device
    n_gpus = torch.cuda.device_count() if "cuda" in device else 0
    print(f"  [gpu] device={device}  n_gpus={n_gpus}", flush=True)
    if device == "cpu":
        raise RuntimeError(
            "CUDA is not available. On Kaggle, enable an accelerator first: "
            "notebook Settings -> Accelerator -> GPU T4 x2, then rerun. "
            "SAM2 fine-tuning on CPU would take days and appear to hang."
        )
    if n_gpus >= 2:
        print(f"  [gpu] {n_gpus} GPUs detected -- folds are sharded one per GPU "
              "unless --gpu-shards 1", flush=True)

    # ── Work dir ──────────────────────────────────────────────────────────
    if args.work_dir:
        work = Path(args.work_dir)
    elif Path("/kaggle/working").is_dir():
        work = Path("/kaggle/working") / SESSION_NAME
    else:
        work = Path(".") / SESSION_NAME
    work.mkdir(parents=True, exist_ok=True)
    ckpt_cache    = work / "pretrained"
    preds_dir     = work / "predictions"
    preds_dir.mkdir(parents=True, exist_ok=True)
    summary_path  = work / "summary_session25.json"
    zip_path      = work / f"SESSION_25_results_seed{args.seed}.zip"

    # Import fold artifacts from earlier Kaggle sessions so completed folds
    # are not retrained. Accepts results zips or directories of fold_* files.
    for src_text in args.resume_from:
        src = Path(src_text)
        imported = _import_resume_artifacts(src, preds_dir, work)
        print(f"  [resume] imported {imported} fold artifact(s) from {src}",
              flush=True)

    # Set aside fold results trained with a different epoch count so folds
    # with different recipes are never pooled into one OOF score.
    for metrics_file in sorted(preds_dir.glob("fold_*_metrics.json")):
        saved_epochs = json.loads(metrics_file.read_text()).get("epochs")
        if saved_epochs is not None and saved_epochs != args.epochs:
            fold_id = metrics_file.name.split("_")[1]
            for stale in preds_dir.glob(f"fold_{fold_id}_*.json"):
                stale.rename(stale.with_suffix(".json.stale"))
            print(f"  [resume] fold {fold_id} was trained for {saved_epochs} "
                  f"epochs but --epochs is {args.epochs}; set aside as "
                  ".stale and will be retrained", flush=True)

    # ── Dataset ───────────────────────────────────────────────────────────
    root = _find_dataset_root(args.data_root)
    names = _load_class_names(root)
    train_items, valid_items, test_items = _discover_dataset(root, names)
    all_items = train_items + valid_items + test_items
    if len(all_items) < 10:
        raise RuntimeError(
            f"Only {len(all_items)} labelled images were found "
            f"under {root}; expected at least 10."
        )
    dataset_evaluation_note = None
    filename_groups = None
    source_manifest = _apply_source_manifest(root, all_items)
    if source_manifest:
        trainval = train_items + valid_items
        trainval_groups = {item["group_id"] for item in trainval}
        test_groups = {item["group_id"] for item in test_items}
        overlap = trainval_groups & test_groups
        if overlap:
            raise RuntimeError(
                f"The dataset manifest places {len(overlap)} source groups in "
                "both train/validation and test splits."
            )
        print(
            f"  [data] {source_manifest['manifest_type']}: "
            f"{source_manifest['source_group_count']} source groups across "
            "train/validation/test",
            flush=True,
        )
    else:
        filename_groups = _apply_filename_groups(all_items)
        if filename_groups:
            kept = [item for item in all_items if not item.get("excluded")]
            cross = filename_groups["cross_split_groups"]
            if cross:
                trainval = kept  # variants cross boundaries -- pool all splits
                dataset_evaluation_note = (
                    "Original train/valid/test splits pooled into source-grouped "
                    "CV: filename-derived groups show augmentation variants cross "
                    "the original boundaries, so the original test split is NOT "
                    "an independent test set. Validation uses one "
                    "original-preferred image per source group."
                )
            else:
                trainval = [
                    item for item in kept if item["split"] != "test"
                ]
                dataset_evaluation_note = (
                    "Filename-derived source groups keep augmentation variants "
                    "in one fold; the received test split is excluded from CV."
                )
            print(
                f"  [data] filename-derived source groups: "
                f"{filename_groups['source_group_count']} groups, "
                f"{filename_groups['kept_images']}/{len(all_items)} images kept "
                f"({filename_groups['excluded_conflict_groups']} groups excluded "
                "for class-set disagreement); "
                f"{cross} groups span more than one original split"
                + (" -- all splits pooled for grouped CV" if cross
                   else " -- test split excluded from CV"),
                flush=True,
            )
        else:
            trainval = train_items + valid_items
            print(
                "  [data] no source manifest -- per-image stratified folds "
                "(test split excluded from CV)",
                flush=True,
            )
    dataset_summary_path = root / "summary.json"
    if dataset_summary_path.is_file():
        dataset_summary = json.loads(
            dataset_summary_path.read_text(encoding="utf-8")
        )
        summary_note = dataset_summary.get("evaluation_note")
        if summary_note:
            dataset_evaluation_note = (
                f"{dataset_evaluation_note} Dataset note: {summary_note}"
                if dataset_evaluation_note else summary_note
            )
            print(f"  [data] evaluation note: {dataset_evaluation_note}", flush=True)

    if not trainval:
        raise RuntimeError(
            f"No labelled train/validation images found under {root}. "
            "Check that YOLO polygon labels are present beside each image."
        )
    pooled = bool(filename_groups and filename_groups["cross_split_groups"])
    print(f"  [data] root={root}  classes={len(names)}"
          f"  cv_images={len(trainval)}"
          + ("" if pooled else f"  test_excluded={len(test_items)}"))
    print(f"  [data] class dist: {dict(Counter(n for it in trainval for n in [names[c] for c in _item_classes(it)]))}")

    # ── Folds ─────────────────────────────────────────────────────────────
    selected = sorted({int(f.strip()) for f in args.folds.split(",") if f.strip()})
    all_folds = _make_folds(trainval, len(names), 5, args.seed)

    accepted_folds = {
        int(f) for f in args.accept_checkpoint_folds.split(",") if f.strip()
    }
    todo = [f for f in selected
            if not (preds_dir / f"fold_{f}_metrics.json").is_file()]
    shards = min(n_gpus if args.gpu_shards == 0 else args.gpu_shards, n_gpus, len(todo))
    if shards > 1:
        _run_gpu_shards(args, todo, work, preds_dir, shards)
        missing = [f for f in selected if not (preds_dir / f"fold_{f}_metrics.json").is_file()]
        if missing:
            print(f"  [shard] folds not finished within budget: {missing} "
                  "-- rerun next session with --resume-from", flush=True)
        selected = [f for f in selected if f not in missing]

    # ── Download SAM2 checkpoint ──────────────────────────────────────────
    checkpoint = _download_checkpoint(ckpt_cache)

    summary: dict = {
        "session": "SESSION_25",
        "runtime_build_id": RUNTIME_BUILD_ID,
        "model": "SAM2 Hiera-Large (ViT-L) frozen image encoder",
        "method": "5-fold OOF CV, GT box prompts (oracle mode), focal+dice loss",
        "metric_type": "segmentation",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_root": str(root),
        "class_names": names,
        "dataset_source_manifest": source_manifest,
        "dataset_source_grouping": filename_groups,
        "dataset_evaluation_note": dataset_evaluation_note,
        "epochs_per_fold": args.epochs,
        "lr_decoder": args.lr_decoder,
        "lr_prompt": args.lr_prompt,
        "trainval_images": len(trainval),
        "seed": args.seed,
        "folds": [],
    }

    def save_summary():
        summary_path.write_text(json.dumps(summary, indent=2))

    oof_gt_images: list = []
    oof_gt_anns:   list = []
    oof_pred_anns: list = []
    fold_map50s:   list = []

    def write_session_artifacts():
        """Refresh summary, merged OOF JSONs, and the results zip.

        Called after every fold so partial results survive a Kaggle
        session kill -- download the zip and pass it to --resume-from
        in the next session.
        """
        oof_pred_path = preds_dir / "SESSION_25_oof.segm.json"
        oof_pred_path.write_text(json.dumps(oof_pred_anns, indent=1))
        oof_gt_path = preds_dir / "SESSION_25_oof_gt.json"
        oof_gt_path.write_text(json.dumps({
            "info": {}, "licenses": [],
            "categories": [{"id": i + 1, "name": n} for i, n in enumerate(names)],
            "images": oof_gt_images, "annotations": oof_gt_anns,
        }, indent=1))
        save_summary()
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(summary_path,  arcname=summary_path.name)
            zf.write(oof_pred_path, arcname=oof_pred_path.name)
            zf.write(oof_gt_path,   arcname=oof_gt_path.name)
            for artifact in sorted(preds_dir.glob("fold_*_*.json")):
                zf.write(artifact, arcname=artifact.name)
            for pattern in ("fold_*/best_epoch.pth", "fold_*/train_complete.json"):
                for artifact in sorted(work.rglob(pattern)):
                    zf.write(artifact, arcname=str(artifact.relative_to(work)))

    for fold_info in all_folds:
        fold = fold_info["fold"]
        if fold not in selected:
            continue
        fold_gt_path      = preds_dir / f"fold_{fold}_oof_gt.json"
        fold_pred_path    = preds_dir / f"fold_{fold}_oof.segm.json"
        fold_metrics_path = preds_dir / f"fold_{fold}_metrics.json"
        if (fold_gt_path.is_file() and fold_pred_path.is_file()
                and fold_metrics_path.is_file()):
            fold_gt   = json.loads(fold_gt_path.read_text())
            metrics   = json.loads(fold_metrics_path.read_text())
            if metrics.get("epochs") is None:
                print(f"  [resume] warning: fold {fold} has no recorded epoch "
                      "count (older run); confirm it matches --epochs",
                      flush=True)
            oof_gt_images.extend(fold_gt["images"])
            oof_gt_anns.extend(fold_gt["annotations"])
            oof_pred_anns.extend(json.loads(fold_pred_path.read_text()))
            fold_map50s.append(metrics["mAP50"])
            summary["folds"].append({"fold": fold, **metrics})
            print(f"  [resume] fold {fold} already complete "
                  f"(mAP50={metrics['mAP50']:.4f}) -- skipping", flush=True)
            continue

        if time.time() - session_start > MAX_SESSION_SECONDS:
            print(f"  [time] {MAX_SESSION_SECONDS/3600:.1f}-h budget -- "
                  f"stopping before fold {fold}", flush=True)
            break

        print(f"\n{'='*60}")
        print(f"  FOLD {fold}  train={len(fold_info['train'])}  val={len(fold_info['valid'])}")
        print(f"{'='*60}", flush=True)

        fold_work = work / f"fold_{fold}"
        img_id_off = fold * 10_000

        # fresh model per fold
        model, predictor = _build_sam2_model(checkpoint, device)
        _freeze_image_encoder(model)
        model.sam_mask_decoder.train()
        model.sam_prompt_encoder.train()

        restored_ckpt = fold_work / "best_epoch.pth"
        train_marker  = fold_work / "train_complete.json"
        if restored_ckpt.is_file() and (train_marker.is_file() or fold in accepted_folds):
            best_ckpt = restored_ckpt
            print(f"  [resume] fold {fold} training already complete -- "
                  "validating only", flush=True)
        else:
            best_ckpt = _train_fold(
                model, predictor, fold_info["train"],
                fold_work, args, device, session_start,
            )
            if not train_marker.is_file():
                print(f"  [time] budget reached during fold {fold} training; "
                      "fold is NOT saved as complete and will rerun in the "
                      "next session.", flush=True)
                del model, predictor
                torch.cuda.empty_cache()
                gc.collect()
                break

        # GT for eval
        gt_imgs, gt_anns = _build_gt_coco(fold_info["valid"], names, img_id_off)
        # OOF predictions
        pred_imgs, pred_anns = _validate_fold(
            model, predictor, best_ckpt,
            fold_info["valid"], names, device, img_id_off,
        )

        oof_gt_images.extend(gt_imgs)
        oof_gt_anns.extend(gt_anns)
        oof_pred_anns.extend(pred_anns)

        metrics = _compute_coco_metrics(gt_imgs, gt_anns, pred_anns, names)
        print(
            f"  Fold {fold} segmentation mAP50-95: "
            f"{metrics['mAP50-95']:.4f}",
            flush=True,
        )
        _print_map50_report(f"Fold {fold}", metrics, names)
        fold_map50s.append(metrics["mAP50"])
        summary["folds"].append({"fold": fold, **metrics})

        # Persist this fold before moving on: GT + preds first, metrics last
        # as the completion marker that resume checks for.
        fold_gt_path.write_text(json.dumps(
            {"images": gt_imgs, "annotations": gt_anns}))
        fold_pred_path.write_text(json.dumps(pred_anns))
        fold_metrics_path.write_text(json.dumps({**metrics, "epochs": args.epochs}))
        write_session_artifacts()

        del model, predictor
        torch.cuda.empty_cache()
        gc.collect()

    # ── Overall OOF ───────────────────────────────────────────────────────
    print(f"\n{'='*76}")
    print("OOF OVERALL EVALUATION")
    oof_metrics = _compute_coco_metrics(oof_gt_images, oof_gt_anns, oof_pred_anns, names)
    summary["oof_metrics"] = oof_metrics
    if fold_map50s:
        summary["cv_map50_mean"] = _stats.mean(fold_map50s)
        summary["cv_map50_std"]  = _stats.stdev(fold_map50s) if len(fold_map50s) > 1 else 0.0

    print(f"\nCV segmentation mAP50 = {summary.get('cv_map50_mean', 0):.4f}"
          f" +/- {summary.get('cv_map50_std', 0):.4f}", flush=True)
    _print_map50_report("Overall OOF", oof_metrics, names)

    # ── Save final artifacts (same helper used after each fold) ───────────
    write_session_artifacts()

    elapsed = (time.time() - session_start) / 3600
    print(f"\n{'='*76}")
    print(f"SESSION 25 COMPLETE -- {elapsed:.2f} h")
    print(f"OOF segmentation mAP50    = {oof_metrics.get('mAP50', 0):.4f}")
    print(f"OOF segmentation mAP50-95 = {oof_metrics.get('mAP50-95', 0):.4f}")
    print(f"Zip saved  -> {zip_path}")
    print(f"{'='*76}", flush=True)
    return 0


if __name__ == "__main__":
    multiprocessing.freeze_support()
    exit_code = main([] if "ipykernel" in sys.modules else None)
    sys.exit(exit_code)


SESSION 25 -- SAM2 ViT-L (frozen encoder) five-fold OOF CV
BUILD: SESSION25-SAM2-6EP-V9-20261018
PYTHON: /usr/bin/python3
[args] {"data_root": "auto", "work_dir": "(default)", "folds": "0,1,2,3,4", "epochs": 6, "seed": 42, "image_size": 1024, "accum_steps": 8, "device": "auto", "resume_from": "(none)", "gpu_shards": 0, "quick": false}
  [deps] current Python missing SAM2 -- bootstrapping ...
  [deps] probe output:
  Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import torch, torchvision, sam2; from sam2.build_sam import build_sam2; print('torch=', torch.__version__, 'cuda=', torch.cuda.is_available(), 'gpus=', torch.cuda.device_count(), 'sam2=OK')
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'sam2'
  [deps] installing isolated SAM2 env (this takes ~10 min) ...
  [env] creating isolated SAM2 environment at /kaggle/working/session25_sam2_env
  [env] stdlib venv failed (Command '['/kaggle/working/session25_sam2_env/bin/python3', 

SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
